# Fine-tune Llama 3.2 3B with MLX

This notebook demonstrates how to fine-tune the Llama 3.2 3B model using Apple's [MLX](https://github.com/ml-explore/mlx) framework on a Mac with Apple Silicon (M-series chips).

We will use the `mlx-lm` library to perform LoRA (Low-Rank Adaptation) fine-tuning on the 4-bit quantized version of the model, which is highly efficient for memory usage (perfect for 16GB-24GB Macs).

## 1. Install Dependencies

In [1]:
%pip install mlx-lm

/Users/akshay/Workspace/slm_bhagwat_gita/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 2. Prepare Dataset
MLX prefers datasets in a specific format (usually training and validation files in a directory). We will take our existing `gita_qna_for_finetune.jsonl` and split it into `train.jsonl` and `valid.jsonl` in a `data/mlx` directory.

In [2]:
import json
import os
import random

# Config
SOURCE_FILE = "data/processed/gita_qna_for_finetune.jsonl"
MLX_DATA_DIR = "data/mlx"
TRAIN_FILE = os.path.join(MLX_DATA_DIR, "train.jsonl")
VALID_FILE = os.path.join(MLX_DATA_DIR, "valid.jsonl")
SPLIT_RATIO = 0.9 # 90% train, 10% validation

# Ensure directory exists
os.makedirs(MLX_DATA_DIR, exist_ok=True)

# Load data
if os.path.exists(SOURCE_FILE):
    with open(SOURCE_FILE, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f]
    
    # Shuffle and split
    random.seed(42)
    random.shuffle(data)
    
    split_idx = int(len(data) * SPLIT_RATIO)
    train_data = data[:split_idx]
    valid_data = data[split_idx:]
    
    # Save splits
    print(f"Saving {len(train_data)} examples to {TRAIN_FILE}...")
    with open(TRAIN_FILE, 'w', encoding='utf-8') as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
            
    print(f"Saving {len(valid_data)} examples to {VALID_FILE}...")
    with open(VALID_FILE, 'w', encoding='utf-8') as f:
        for item in valid_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
            
    print("Done!")
else:
    print(f"ERROR: Source file {SOURCE_FILE} not found. Please ensure the path is correct.")

Saving 4174 examples to data/mlx/train.jsonl...
Saving 464 examples to data/mlx/valid.jsonl...
Done!


## 3. Configuration
We use the **Llama-3.2-3B-Instruct-4bit** model. It's lightweight (~2GB) and highly capable.
Alternatively, for more power (but more memory), you can use `mlx-community/Meta-Llama-3.1-8B-Instruct-4bit`.

In [3]:
MODEL_NAME = "mlx-community/Llama-3.2-3B-Instruct-4bit"
# MODEL_NAME = "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit" # Alternative for 8B model

## 4. Run Fine-Tuning
We run the fine-tuning using the `mlx_lm.lora` command line tool directly from the notebook.

**Parameters:**
- `--model`: The model to fine-tune.
- `--data`: Path to the directory containing train.jsonl and valid.jsonl.
- `--train`: Flag to start training.
- `--iters`: Number of training iterations (steps). Start with 600 for a quick test, increase for better results.
- `--batch-size`: Batch size (4 is usually safe for 3B/8B on >16GB Macs).
- `--num-layers`: Number of layers to train (16 is a good default).
- `--adapter-path`: Where to save the adapters.

In [5]:
# Run training (This may take some time!)
!mlx_lm.lora \
    --model {MODEL_NAME} \
    --data {MLX_DATA_DIR} \
    --train \
    --iters 20 \
    --batch-size 4 \
    --num-layers 16 \
    --adapter-path "adapters"

Loading pretrained model
Fetching 6 files: 100%|████████████████████████| 6/6 [00:00<00:00, 56552.41it/s]
Loading datasets
Training
Trainable parameters: 0.216% (6.947M/3212.750M)
Starting training..., iters: 20
Calculating loss...: 100%|██████████████████████| 25/25 [00:49<00:00,  2.00s/it]
Iter 1: Val loss 3.675, Val took 49.892s
Iter 10: Train loss 2.549, Learning Rate 1.000e-05, It/sec 0.271, Tokens/sec 158.513, Trained Tokens 5845, Peak mem 5.839 GB
Calculating loss...: 100%|██████████████████████| 25/25 [00:56<00:00,  2.27s/it]
Iter 20: Val loss 1.509, Val took 56.679s
Iter 20: Train loss 1.521, Learning Rate 1.000e-05, It/sec 0.260, Tokens/sec 150.707, Trained Tokens 11644, Peak mem 6.408 GB
Saved final weights to adapters/adapters.safetensors.


## 5. Test Inference
Now we can test the fine-tuned model by loading the adapters we just created.

In [6]:
from mlx_lm import load, generate

# Load model and tokenizer with adapters
model, tokenizer = load(MODEL_NAME, adapter_path="adapters")

# Define a prompt template (Standard Llama 3)
prompt_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a wise teacher drawing from Bhagavad Gita.<|eot_id|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

query = "Using Chapter 2, Verse 47, answer: What is the nature of duty?"
prompt = prompt_template.format(query)

response = generate(model, tokenizer, prompt=prompt, verbose=True)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

The nature of duty is to fulfill one's role in life, which is a means to achieve spiritual growth and self-realization. It is a way to maintain the balance of life, ensuring that one's actions are guided by wisdom, compassion, and a sense of duty to others.
Prompt: 46 tokens, 68.190 tokens-per-sec
Generation: 58 tokens, 42.179 tokens-per-sec
Peak memory: 2.012 GB


## 6. Fuse and Save
To use the model easily with valid GGUF tools or just as a standalone MLX model, we can "fuse" the adapters into the base model.

MLX includes a `fuse` command to do this.

In [7]:
!mlx_lm.fuse \
    --model {MODEL_NAME} \
    --adapter-path "adapters" \
    --save-path "models/gita-llama-3.2-3b-fused"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading pretrained model
Fetching 6 files: 100%|███████████████████████| 6/6 [00:00<00:00, 113359.57it/s]
README.md: 16.3kB [00:00, 31.5MB/s]


## 7. Next Steps
The fused model is now saved in `models/gita-llama-3.2-3b-fused`. You can:
1. Use it directly with `mlx-lm` in your scripts.
2. Convert it to GGUF using `llama.cpp`'s conversion scripts if you specifically need GGUF format (MLX models are already highly optimized for Mac).